# Ejercicio: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [ ]:
!pip install beir rank_bm25

In [16]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

In [17]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

'../data/beir_datasets\\scifact'

In [18]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

100%|██████████| 5183/5183 [00:00<00:00, 102902.49it/s]


In [19]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [20]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [21]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [22]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [23]:
from rank_bm25 import BM25Okapi
from tqdm import tqdm

# texto de cada documento
corpus_texts = [
    (doc["title"] + " " + doc["text"]).lower()
    for doc in corpus.values()
]

doc_ids = list(corpus.keys())

# tokenización sencilla
tokenized_corpus = [
    doc.split()
    for doc in corpus_texts
]

bm25 = BM25Okapi(tokenized_corpus)

In [24]:
results_bm25 = {}

for qid, query in tqdm(queries.items()):

    query_tokens = query.lower().split()

    scores = bm25.get_scores(query_tokens)

    ranking = {
        doc_ids[i]: float(scores[i])
        for i in range(len(doc_ids))
    }

    results_bm25[qid] = ranking

100%|██████████| 300/300 [00:07<00:00, 42.30it/s]


In [25]:
from beir.retrieval.evaluation import EvaluateRetrieval

k_values = [10]

ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
    qrels,
    results_bm25,
    k_values
)

print("Recall@10 =", recall["Recall@10"])
print("nDCG@10   =", ndcg["NDCG@10"])
print("MAP@10    =", _map["MAP@10"])

Recall@10 = 0.68617
nDCG@10   = 0.5597
MAP@10    = 0.51473


In [26]:
qid = "133"

top10 = sorted(
    results_bm25[qid].items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

pd.DataFrame(top10, columns=["doc_id", "bm25_score"])

,doc_id,bm25_score
0,26688294,55.121353
1,9507605,50.471045
2,37964706,49.902940
3,5270265,46.273261
4,12785130,46.220216
5,12640810,45.945610
6,30861948,45.683857
7,86694016,45.608148
8,17934082,45.531225
9,6969753,44.885877


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [27]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

C:\Users\Michael\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Michael\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading

In [32]:
from tqdm import tqdm

TOP_K = 20

results_ce = {}

for qid in tqdm(queries):

    query = queries[qid]

    candidates = sorted(
        results_bm25[qid].items(),
        key=lambda x: x[1],
        reverse=True
    )[:TOP_K]

    pairs = []

    for doc_id, _ in candidates:

        text = (
            corpus[doc_id]["title"]
            + " "
            + corpus[doc_id]["text"]
        )

        pairs.append((query, text))

    ce_scores = cross_encoder.predict(
        pairs,
        show_progress_bar=False
    )

    results_ce[qid] = {
        doc_id: float(score)
        for (doc_id, _), score in zip(candidates, ce_scores)
    }

100%|██████████| 300/300 [06:19<00:00,  1.27s/it]


In [33]:
from beir.retrieval.evaluation import EvaluateRetrieval

ndcg_ce, map_ce, recall_ce, precision_ce = EvaluateRetrieval.evaluate(
    qrels,
    results_ce,
    [10]
)

print("Cross Encoder")
print("nDCG@10 :", ndcg_ce["NDCG@10"])
print("MAP@10  :", map_ce["MAP@10"])
print("Recall@10 :", recall_ce["Recall@10"])

Cross Encoder
nDCG@10 : 0.63704
MAP@10  : 0.60404
Recall@10 : 0.72033


In [34]:
qid = "133"

bm25_top10 = [
    doc
    for doc, _ in sorted(
        results_bm25[qid].items(),
        key=lambda x: x[1],
        reverse=True
    )[:10]
]

ce_top10 = [
    doc
    for doc, _ in sorted(
        results_ce[qid].items(),
        key=lambda x: x[1],
        reverse=True
    )[:10]
]

comparison = pd.DataFrame({
    "BM25": bm25_top10,
    "CrossEncoder": ce_top10
})

comparison

,BM25,CrossEncoder
0,26688294,12640810
1,9507605,6969753
2,37964706,9507605
3,5270265,86694016
4,12785130,19752008
5,12640810,17934082
6,30861948,9063688
7,86694016,4399311
8,17934082,37964706
9,6969753,23076291


In [35]:
qid = "133"

bm25_rank = {
    doc: rank + 1
    for rank, (doc, _) in enumerate(
        sorted(
            results_bm25[qid].items(),
            key=lambda x: x[1],
            reverse=True
        )[:10]
    )
}

ce_rank = {
    doc: rank + 1
    for rank, (doc, _) in enumerate(
        sorted(
            results_ce[qid].items(),
            key=lambda x: x[1],
            reverse=True
        )[:10]
    )
}

changes = []

for doc in set(bm25_rank.keys()).union(ce_rank.keys()):

    changes.append({
        "doc_id": doc,
        "rank_bm25": bm25_rank.get(doc),
        "rank_ce": ce_rank.get(doc)
    })

changes_df = pd.DataFrame(changes)

changes_df[
    changes_df["rank_bm25"] != changes_df["rank_ce"]
].sort_values("rank_ce")

,doc_id,rank_bm25,rank_ce
13,12640810,6.0,1.0
8,6969753,10.0,2.0
10,9507605,2.0,3.0
3,86694016,8.0,4.0
7,19752008,NaN,5.0
4,17934082,9.0,6.0
12,9063688,NaN,7.0
0,4399311,NaN,8.0
2,37964706,3.0,9.0
5,23076291,NaN,10.0


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [38]:
def get_rank_changes(qid, k=10):

    bm25_rank = {
        doc: rank + 1
        for rank, (doc, _) in enumerate(
            sorted(results_bm25[qid].items(), key=lambda x: x[1], reverse=True)[:k]
        )
    }

    ce_rank = {
        doc: rank + 1
        for rank, (doc, _) in enumerate(
            sorted(results_ce[qid].items(), key=lambda x: x[1], reverse=True)[:k]
        )
    }

    docs = set(bm25_rank.keys()).union(ce_rank.keys())

    df = pd.DataFrame([
        {"doc_id": d, "rank_bm25": bm25_rank.get(d), "rank_ce": ce_rank.get(d)}
        for d in docs
    ])

    df["changed"] = df["rank_bm25"] != df["rank_ce"]
    return df

In [41]:
# Ejemplo puntual 
changes_133 = get_rank_changes("133")
changes_133[changes_133["changed"]].sort_values("rank_ce")

,doc_id,rank_bm25,rank_ce,changed
13,12640810,6.0,1.0,True
8,6969753,10.0,2.0,True
10,9507605,2.0,3.0,True
3,86694016,8.0,4.0,True
7,19752008,NaN,5.0,True
4,17934082,9.0,6.0,True
12,9063688,NaN,7.0,True
0,4399311,NaN,8.0,True
2,37964706,3.0,9.0,True
5,23076291,NaN,10.0,True


In [42]:
# Generalizando a TODAS las queries: cuántos docs cambian de posición en el top-10
summary_rows = []

for qid in tqdm(results_ce.keys()):
    df = get_rank_changes(qid)
    n_changed = df["changed"].sum()
    n_new_in_top10 = df["rank_bm25"].isna().sum()      
    n_dropped_from_top10 = df["rank_ce"].isna().sum() 

    summary_rows.append({
        "query_id": qid,
        "docs_distintos": len(df),
        "docs_que_cambiaron_posicion": n_changed,
        "docs_nuevos_en_top10": n_new_in_top10,
        "docs_que_salieron_top10": n_dropped_from_top10
    })

rank_changes_summary = pd.DataFrame(summary_rows)
rank_changes_summary

100%|██████████| 300/300 [00:00<00:00, 387.32it/s]


,query_id,docs_distintos,docs_que_cambiaron_posicion,docs_nuevos_en_top10,docs_que_salieron_top10
0,1,14,13,4,4
1,3,12,10,2,2
2,5,15,14,5,5
3,13,13,13,3,3
4,36,14,14,4,4
...,...,...,...,...,...
295,1379,14,13,4,4
296,1382,14,13,4,4
297,1385,13,11,3,3
298,1389,14,12,4,4


In [43]:
print("Promedio de docs que cambian de posición por query:",
      rank_changes_summary["docs_que_cambiaron_posicion"].mean())

print("Queries donde el top-10 cambió en al menos 1 documento:",
      (rank_changes_summary["docs_que_cambiaron_posicion"] > 0).sum(),
      "de", len(rank_changes_summary))

Promedio de docs que cambian de posición por query: 12.126666666666667
Queries donde el top-10 cambió en al menos 1 documento: 300 de 300


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [44]:
comparison_metrics = pd.DataFrame({
    "BM25": {
        "nDCG@10": ndcg["NDCG@10"],
        "MAP@10": _map["MAP@10"],
        "Recall@10": recall["Recall@10"],
    },
    "CrossEncoder (LTR)": {
        "nDCG@10": ndcg_ce["NDCG@10"],
        "MAP@10": map_ce["MAP@10"],
        "Recall@10": recall_ce["Recall@10"],
    }
}).T

comparison_metrics["mejora_%"] = (
    (comparison_metrics["nDCG@10"] - comparison_metrics["nDCG@10"].iloc[0])
    / comparison_metrics["nDCG@10"].iloc[0] * 100
)

comparison_metrics

,nDCG@10,MAP@10,Recall@10,mejora_%
BM25,0.55970,0.51473,0.68617,0.000000
CrossEncoder (LTR),0.63704,0.60404,0.72033,13.818117
